In [1]:
%load_ext autoreload
%autoreload 2

from datetime import datetime, timedelta, date
from functools import partial
import numpy as np
import polars as pl

from okx.store import OrderbookStore
from okx.recipes.options import prepare_options, build_forwards_options_comparison
from okx.recipes.forwards import build_forwards_pchip, build_forwards_kalman, prepare_pillars

from okx.features import add_tenor, parse_option
from okx.recipes.helpers import early_roll

In [2]:
store = OrderbookStore(
    data_root="data/okx",
    manifest_path="data/okx/manifest.sqlite",
    batch_days=5
)

dates = [date(2025, 9, 1)]

In [ ]:
early_roll_fn = early_roll(min_time_to_expiry_hours=2.0)

start_time = datetime.now()

options_df = store.get(
    inst_family='BTC-USD',
    inst_type='OPTION',
    dates=dates,
    depth=1,
    features=['trim', 'strip']
)

time_1 = datetime.now()
print(f"Time taken to fetch options: {time_1 - start_time}")

options_df = add_tenor(options_df, inst_type='OPTION')
time_2 = datetime.now()
print(f"Time taken to add tenor: {time_2 - time_1}")

options_df = early_roll_fn(options_df)
time_3 = datetime.now()
print(f"Time taken to apply early roll: {time_3 - time_2}")

options_df = parse_option(options_df)
time_4 = datetime.now()
print(f"Time taken to parse options: {time_4 - time_3}")

Time taken to fetch options: 0:00:00.050789


AttributeError: 'DataFrame' object has no attribute 'collect'

In [ ]:
    # Build feature list
    options_features = ['trim', 'strip']
    if binning:
        options_features.extend(['bin', finalize_binning])
    options_features.extend(['tenor', early_roll(min_time_to_expiry_hours), 'parse_option'])
    
    # Load options orderbook
    df_options = store.get(
        inst_type='OPTION',
        inst_family=inst_family,
        dates=dates,
        depth=1,
        binning=binning,
        features=options_features,
        cache_name=cache_name,
    ).collect()